# Analysis — RAG strategy comparison

Loads the judge eval files from `results/` and produces the tables and figures:
answer rate, and per-metric scores in two forms —

- **Conditional** (zeros excluded): quality *when the strategy attempts an answer* — matches the paper's tables.
- **Unconditional** (refusals = 0): quality *over all queries* — the fair cross-strategy comparison.

Run top-to-bottom. Requires that `scripts.judge` has been run for each strategy.


In [ ]:
import sys, os
# allow importing the rag package from the project root
sys.path.insert(0, os.path.abspath(".."))

import pandas as pd
import matplotlib.pyplot as plt

from rag import analysis as A
from rag.config import cfg

pd.set_option("display.width", 120)
print("Results dir:", cfg.paths.results_dir)
print("Strategies found:", A.discover_strategies())


In [ ]:
# Aggregate everything into one tidy DataFrame
df = A.aggregate()
df.head(20)


## Answer rate by strategy and query type

In [ ]:
ar = A.answer_rate_table(df)
display(ar)

ax = ar.plot(kind="bar", figsize=(10,4), rot=30)
ax.set_ylabel("Answer rate (%)")
ax.set_title("Answer rate (is_answer = true) by strategy")
ax.set_ylim(0, 105)
ax.legend(title="query type")
plt.tight_layout()
plt.savefig(os.path.join(cfg.paths.results_dir, "fig_answer_rate.png"), dpi=150)
plt.show()


## Overall score — conditional (zeros excluded, matches paper)

In [ ]:
overall_cond = A.overall_table(df, "cond")
display(overall_cond)


## Overall score — unconditional (refusals = 0, fair comparison)

In [ ]:
overall_uncond = A.overall_table(df, "uncond")
display(overall_uncond)

# Side-by-side of the two means so the gap (caused by refusals) is visible
compare = pd.DataFrame({
    "conditional_mean": overall_cond["mean"],
    "unconditional_mean": overall_uncond["mean"],
})
compare["refusal_penalty"] = (compare["conditional_mean"] - compare["unconditional_mean"]).round(2)
display(compare.sort_values("unconditional_mean", ascending=False))


## Per-metric breakdown (conditional)

In [ ]:
def per_metric_table(df, query_type, kind="cond"):
    sub = df[df.query_type == query_type].set_index("strategy")
    cols = [f"{m}_{kind}" for m in A.METRICS]
    out = sub[cols].copy()
    out.columns = [m.replace("_score","").replace("_"+kind,"") for m in cols]
    return out.round(2)

for qt in ["problem", "method"]:
    print(f"=== {qt.upper()} queries (conditional) ===")
    display(per_metric_table(df, qt, "cond"))


## Per-metric chart (one panel per query type)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15,5), sharey=True)
for ax, qt in zip(axes, ["problem", "method"]):
    t = per_metric_table(df, qt, "cond")
    t.T.plot(kind="bar", ax=ax, rot=30)
    ax.set_title(f"{qt.capitalize()} queries — per-metric (conditional)")
    ax.set_ylabel("Score (1-5)")
    ax.set_ylim(0, 5.2)
    ax.legend(title="strategy", fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(cfg.paths.results_dir, "fig_per_metric.png"), dpi=150)
plt.show()


## Export tables as CSV

Saved next to the figures in `results/`, ready to drop into the paper or share.

In [ ]:
overall_cond.to_csv(os.path.join(cfg.paths.results_dir, "table_overall_conditional.csv"))
overall_uncond.to_csv(os.path.join(cfg.paths.results_dir, "table_overall_unconditional.csv"))
A.answer_rate_table(df).to_csv(os.path.join(cfg.paths.results_dir, "table_answer_rate.csv"))
df.to_csv(os.path.join(cfg.paths.results_dir, "metrics_full.csv"), index=False)
print("Wrote CSVs and PNGs to", cfg.paths.results_dir)
